# RHOAI documentation retrieval pipeline

This notebook demonstrates the full enterprise RAG retrieval flow from the
[Build an enterprise RAG system with OGX](https://developers.redhat.com/blog/2025/06/26/build-enterprise-rag-system-ogx)
blog post, applied to the **RHOAI product documentation corpus**.

The pipeline implements four stages:

1. **Metadata extraction** -- use the LLM to extract a structured topic filter
   from a natural-language question.
2. **Hybrid search** -- combine dense vector search with keyword search,
   filtered by the extracted metadata.
3. **Neural reranking** -- send the top candidates to a Qwen3 cross-encoder
   for more precise relevance scoring.
4. **Grounded answer generation** -- pass the reranked context to Nemotron
   through Models-as-a-Service to produce a cited, documentation-grounded
   answer.

## 1. Configure runtime endpoints

Connect to Llama Stack, the MaaS Nemotron generation endpoint, and the
neural reranker. All endpoints are injected as environment variables by
the GitOps-managed workbench.

In [ ]:
import json
import os
import ssl
import urllib.request
import urllib.error
from llama_stack_client import LlamaStackClient

base_url = os.environ.get("LLAMA_STACK_BASE_URL",
    "http://lsd-enterprise-rag-service.enterprise-rag.svc.cluster.local:8321")
reranker_base_url = os.environ.get("RHOAI_STAGE230_RERANKER_BASE_URL", base_url)
generation_base_url = os.environ.get("RHOAI_STAGE230_GENERATION_BASE_URL")
generation_api_key = os.environ.get("RHOAI_STAGE230_GENERATION_API_KEY")
generation_model = os.environ.get("RHOAI_STAGE230_GENERATION_MODEL", "nemotron-3-nano-30b-a3b")
reranker_model = os.environ.get("RHOAI_STAGE230_RERANKER_MODEL", "vllm-reranker/qwen3-reranker")

client = LlamaStackClient(base_url=base_url)

print("Llama Stack:", base_url)
print("Generation:", generation_base_url)
print("Generation model:", generation_model)
print("Reranker model:", reranker_model)

## 2. Find the RHOAI product documentation vector store

In [ ]:
VECTOR_STORE_NAME = os.environ.get(
    "RHOAI_STAGE230_RHOAI_DOCS_VECTOR_STORE",
    "stage230-rhoai-34-product-docs",
)

stores = client.vector_stores.list()
store_list = list(stores) if not isinstance(stores, list) else stores

vector_store_id = None
available_stores = []
for store in store_list:
    name = getattr(store, 'name', None)
    sid = getattr(store, 'id', None)
    available_stores.append(name)
    if name == VECTOR_STORE_NAME:
        vector_store_id = sid
        print(f"Found vector store: {name} (ID: {sid})")
        break

if not vector_store_id:
    raise RuntimeError(
        f"Vector store '{VECTOR_STORE_NAME}' not found. "
        f"Available: {available_stores}. "
        f"Set RHOAI_STAGE230_RHOAI_DOCS_VECTOR_STORE to use a different store "
        f"(e.g. 'stage230-rhoai-34-product-docs-dev' for notebook data)."
    )

## 3. Define the metadata extraction step

Following the OGX blog post pattern, we use the LLM to extract structured
metadata from a natural language query. For RHOAI product docs, we extract
a `topic` filter (e.g., `llama_stack_rag`, `autorag`, `guardrails`).

In [ ]:
VALID_TOPICS = [
    "llama_stack_rag", "autorag", "ragas_evaluation", "evalhub",
    "lm_eval", "risk_assessment", "guardrails", "ai_pipelines",
    "docling_data_prep",
]

def api_url(base, path):
    base = base.rstrip("/")
    path = path.lstrip("/")
    if base.endswith("/v1") and path.startswith("v1/"):
        path = path[3:]
    return f"{base}/{path}"

def http_json(method, url, payload=None, api_key=None):
    data = json.dumps(payload).encode() if payload else None
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    req = urllib.request.Request(url, data=data, method=method, headers=headers)
    ctx = ssl._create_unverified_context()
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            return json.loads(resp.read().decode())
    except urllib.error.HTTPError as exc:
        body = exc.read().decode(errors="replace")
        raise RuntimeError(f"{method} {url} -> HTTP {exc.code}: {body[:500]}") from exc

def extract_metadata_filter(query):
    """Use Nemotron to extract a topic filter from the user's question."""
    import re
    payload = {
        "model": generation_model,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You extract RHOAI documentation topic filters. Return only compact JSON "
                    f'like {{"topic":"guardrails"}}. Valid topics: {", ".join(VALID_TOPICS)}. '
                    "Use null if no specific topic is implied."
                ),
            },
            {"role": "user", "content": query},
        ],
        "temperature": 0,
        "max_tokens": 64,
    }
    response = http_json("POST", api_url(generation_base_url, "/v1/chat/completions"),
                         payload, api_key=generation_api_key)
    content = response["choices"][0]["message"].get("content") or ""
    match = re.search(r"\{.*\}", content, re.DOTALL)
    if not match:
        return {}, None
    parsed = json.loads(match.group(0))
    topic = parsed.get("topic")
    if topic and topic in VALID_TOPICS:
        return {"type": "eq", "key": "topic", "value": topic}, topic
    return {}, None

print("Metadata extraction function ready")
print(f"Valid topics: {VALID_TOPICS}")

## 4. Ask a question and extract the metadata filter

We extract a `topic` filter from the user's RHOAI documentation question
(e.g., `llama_stack_rag`) to narrow retrieval to the relevant guide.

In [ ]:
query = "How does OpenShift AI use Llama Stack, vector stores, and hybrid search for RAG?"

filters, extracted_topic = extract_metadata_filter(query)
print(f"Query: {query}")
print(f"Extracted topic: {extracted_topic}")
print(f"Metadata filter: {json.dumps(filters, indent=2)}")

## 5. Run hybrid search with metadata filtering

Using `search_mode='hybrid'` combines dense vector search with keyword
search, filtered by the extracted metadata.

In [ ]:
search_kwargs = {
    "vector_store_id": vector_store_id,
    "query": query,
    "search_mode": "hybrid",
    "max_num_results": 8,
}
if filters:
    search_kwargs["filters"] = filters

results = client.vector_stores.search(**search_kwargs)
candidates = list(results) if not isinstance(results, list) else results
if hasattr(results, 'data'):
    candidates = list(results.data)

def get_text(hit):
    content = getattr(hit, 'content', None) or (hit.get('content') if isinstance(hit, dict) else None)
    if isinstance(content, list):
        parts = []
        for p in content:
            t = getattr(p, 'text', None) or (p.get('text') if isinstance(p, dict) else None)
            if t:
                parts.append(str(t))
        if parts:
            return "\n".join(parts)
    for key in ('text', 'chunk_text'):
        val = getattr(hit, key, None) or (hit.get(key) if isinstance(hit, dict) else None)
        if val:
            return str(val)
    return str(hit)

print(f"Search mode: hybrid")
print(f"Filter: topic={extracted_topic}")
print(f"Candidates returned: {len(candidates)}\n")

for i, hit in enumerate(candidates[:5]):
    text = get_text(hit)
    attrs = getattr(hit, 'attributes', None) or (hit.get('attributes') if isinstance(hit, dict) else {})
    guide = (attrs.get('guide_slug', '?') if isinstance(attrs, dict) else '?')
    score = getattr(hit, 'score', None) or (hit.get('score') if isinstance(hit, dict) else None)
    print(f"[{i+1}] guide={guide}, score={score}")
    print(f"    {text[:150]}...")
    print()

## 6. Rerank candidates with the Qwen3 cross-encoder

Following the OGX blog post, we send the top candidates to the neural
reranker. The cross-encoder processes each query-document pair together
to produce more accurate relevance scores.

In [ ]:
documents = [get_text(hit)[:1200] for hit in candidates[:4]]
typed_items = [{"type": "text", "text": doc} for doc in documents]

rerank_payload = {
    "model": reranker_model,
    "query": query,
    "items": typed_items,
    "max_num_results": min(3, len(documents)),
}

rerank_url = api_url(reranker_base_url, "/v1alpha/inference/rerank")
try:
    rerank_response = http_json("POST", rerank_url, rerank_payload)
except Exception:
    rerank_url = api_url(reranker_base_url, "/v1/rerank")
    rerank_payload_alt = {
        "model": reranker_model,
        "query": query,
        "documents": documents,
        "top_n": min(3, len(documents)),
    }
    rerank_response = http_json("POST", rerank_url, rerank_payload_alt)

raw_results = rerank_response.get("results") or rerank_response.get("data") or rerank_response.get("rankings") or []

ranked = []
for r in raw_results:
    idx = int(r.get("index", r.get("document_index", 0)))
    score = r.get("relevance_score", r.get("score", r.get("logit")))
    ranked.append({"index": idx, "score": score, "text": documents[idx] if idx < len(documents) else ""})

print(f"Reranker endpoint: {rerank_url}")
print(f"Reranked {len(ranked)} candidates:\n")
for i, item in enumerate(ranked):
    print(f"[{i+1}] score={item['score']:.4f}" if isinstance(item['score'], (int, float)) else f"[{i+1}] score={item['score']}")
    print(f"    {item['text'][:150]}...")
    print()

## 7. Generate a grounded answer with Nemotron

The final step sends the reranked context to Nemotron through MaaS,
instructing it to answer using only the retrieved official Red Hat
documentation.

In [ ]:
context = "\n\n".join(f"[{i+1}] {item['text']}" for i, item in enumerate(ranked))

answer_payload = {
    "model": generation_model,
    "messages": [
        {
            "role": "system",
            "content": (
                "You explain Red Hat OpenShift AI product capabilities to a technical demo audience. "
                "Answer using only the retrieved official Red Hat documentation context. "
                "Mention the relevant product component names. If the context is insufficient, say so."
            ),
        },
        {"role": "user", "content": f"Question: {query}\n\nRetrieved context:\n{context}"},
    ],
    "temperature": 0,
    "max_tokens": 700,
}

answer_response = http_json("POST", api_url(generation_base_url, "/v1/chat/completions"),
                            answer_payload, api_key=generation_api_key)

answer = answer_response["choices"][0]["message"].get("content", "")
if isinstance(answer, list):
    answer = "\n".join(str(p.get("text", "")) for p in answer if isinstance(p, dict))

print(f"Question: {query}\n")
print(f"Topic filter: {extracted_topic}")
print(f"Candidates retrieved: {len(candidates)}")
print(f"Candidates reranked: {len(ranked)}\n")
print("=" * 60)
print("ANSWER:")
print("=" * 60)
print(answer.strip())

## 8. Try more questions

Change the query below and re-run to explore different RHOAI product
documentation topics.

In [ ]:
query2 = "What role do AI Pipelines and Kubeflow Pipelines play in OpenShift AI?"

filters2, topic2 = extract_metadata_filter(query2)
print(f"Query: {query2}")
print(f"Extracted topic: {topic2}")
print(f"Filter: {json.dumps(filters2)}\n")

search_kwargs2 = {
    "vector_store_id": vector_store_id,
    "query": query2,
    "search_mode": "hybrid",
    "max_num_results": 8,
}
if filters2:
    search_kwargs2["filters"] = filters2

results2 = client.vector_stores.search(**search_kwargs2)
candidates2 = list(results2) if not isinstance(results2, list) else results2
if hasattr(results2, 'data'):
    candidates2 = list(results2.data)

docs2 = [get_text(h)[:1200] for h in candidates2[:4]]
typed2 = [{"type": "text", "text": d} for d in docs2]
rr2 = http_json("POST", api_url(reranker_base_url, "/v1alpha/inference/rerank"), {
    "model": reranker_model, "query": query2, "items": typed2, "max_num_results": 3,
})
raw2 = rr2.get("results") or rr2.get("data") or rr2.get("rankings") or []
ranked2 = [{"index": int(r.get("index",0)), "score": r.get("relevance_score", r.get("score")),
            "text": docs2[int(r.get("index",0))]} for r in raw2]

ctx2 = "\n\n".join(f"[{i+1}] {item['text']}" for i, item in enumerate(ranked2))
ans2 = http_json("POST", api_url(generation_base_url, "/v1/chat/completions"), {
    "model": generation_model,
    "messages": [
        {"role": "system", "content": "You explain Red Hat OpenShift AI product capabilities to a technical demo audience. Answer using only the retrieved official Red Hat documentation context. Mention the relevant product component names. If the context is insufficient, say so."},
        {"role": "user", "content": f"Question: {query2}\n\nRetrieved context:\n{ctx2}"},
    ],
    "temperature": 0, "max_tokens": 700,
}, api_key=generation_api_key)

answer2 = ans2["choices"][0]["message"].get("content", "")
if isinstance(answer2, list):
    answer2 = "\n".join(str(p.get("text", "")) for p in answer2 if isinstance(p, dict))

print("=" * 60)
print("ANSWER:")
print("=" * 60)
print(answer2.strip())

---

## Summary

This notebook demonstrates the four-stage enterprise RAG retrieval
pattern applied to the **RHOAI product documentation corpus**:

| Stage | What happens | Component |
|-------|-------------|-----------|
| **Metadata extraction** | LLM extracts a `topic` filter from the question | Nemotron via MaaS |
| **Hybrid search** | Dense + keyword search with metadata filtering | Llama Stack + pgvector |
| **Neural reranking** | Cross-encoder rescores query-document pairs | Qwen3 reranker via vLLM |
| **Grounded generation** | Answer grounded in retrieved documentation | Nemotron via MaaS |

The same pipeline is automated through:

- The **Stage 230 Streamlit chatbot** for interactive use.
- The **KFP pipeline** for batch ingestion and retrieval evaluation.